# How Diversified is an S&P 500 ETF?
## Technology Concentration and Revenue Growth in the S&P 500 at the End of 2024

**Author:** Tobias Frei · **Course:** AI Mastery Capstone, Project 2 · **Date:** 2026-11-21

**Dataset:** sp500_companies.csv from S&P 500 Stocks by Larxel, Version 1023, Kaggle, CC0 license
(https://www.kaggle.com/datasets/andrewmvd/sp-500-stocks). 

The file describes the companies in the S&P 500 index on 2024-12-20, one row per listed ticker, with sector, market value, revenue growth and further company data. The Kaggle page does not name the source, but the columns match the data of Yahoo Finance.
It is a snapshot, so every value refers to the state of the companies at the end of 2024.

"Daily updated" means that the dataset was rebuilt every day until December 2024, not that this file contains daily data. The file has no date column. I took 2024-12-20 as the date of the snapshot because it is the last trading day in
the price files of the same version (Version 1023, published 2024-12-23).


Note: Finding the right dataset for this project was a challenge I also investigated several different Kaggle datasets, but some where to broad and deep so I decided to go with this one, since it ticked all the requirements and is suitable to answer my research questions.

### Motivation
An S&P 500 ETF is often called a safe long-term investment, because you own 500 companies. But the index is weighted by market value, so a few big technology companies get a large part of the money. In this notebook I first look at how concentrated the index is. Then I test if tech companies really grow their revenue faster than the rest, which would be one reason for their big weight. 
Disclaimer: This is not investment advice.

### Analysis Plan
I created this plan prior to the comparison of groups, so the results dont influence my hypothesis. This follows the advice from (Lusa et al., 2024) to conduct data analysis initially.

### Research Flow

- Descriptive Statistics: What share of the total market value do the sectors and the 10 biggest companies posses, comparing it to their share of all the companies.
- Test Questions: Analysis the question if the mean revenue growth of tech companies differ from the mean of all other companies.
- Outcome: Will be calculated in Revenuegrowth in %
- Groups: Will filter the companies: Technology vs. all other sectors, one row per ticker.
- Hypotheses: H0: μ_Tech = μ_Other_Sectors: They are the same: H1: μ_Tech ≠ μ_Other_Sectors (two-sided) They are significantly different testing both lower and higher contribution.
- Significance Level: I am using an alpha of 0.05
- Testing: I will be using a t-test
- Robustness Checking: I will use the bootstrap method with 95% intervals for the differences in means and in the medians with a t-test to investigate further.

All the decisions I make along the way will be documented.

### Table of Content
1. Setup
2. Ingestion & Dataset Description
3. Data Investigation
5. Data Cleaning
6. Descriptive Statistics
7. Visualization
8. Hypothesis Testing
9. Summary




---

## 1. Setup

In [17]:
# Libraries Import
import sys
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
from scipy import stats


In [18]:
# Setting up configurations 
DATA = "sp500_companies.csv"
ALPHA = 0.05
SEED = 42

In [19]:
# Checking versions to avoid conflicts
print(f"Python {sys.version.split()[0]}, pandas {pd.__version__}, NumPy {np.__version__}, " f"SciPy {scipy.__version__}, Matplotlib {matplotlib.__version__}, seaborn {sns.__version__}")

Python 3.12.7, pandas 3.1.1, NumPy 2.5.5, SciPy 1.18.1, Matplotlib 3.11.2, seaborn 0.13.2


---

## 2. Ingestion & Dataset Description

In [20]:
# Loading the raw data
df_raw = pd.read_csv(DATA)
df_raw.head()

,Exchange,Symbol,Shortname,Longname,Sector,Industry,Currentprice,Marketcap,Ebitda,Revenuegrowth,City,State,Country,Fulltimeemployees,Longbusinesssummary,Weight
0,NMS,AAPL,Apple Inc.,Apple Inc.,Technology,Consumer Electronics,254.49,3846819807232,1.346610e+11,0.061,Cupertino,CA,United States,164000.0,"Apple Inc. designs, manufactures, and markets ...",0.069209
1,NMS,NVDA,NVIDIA Corporation,NVIDIA Corporation,Technology,Semiconductors,134.70,3298803056640,6.118400e+10,1.224,Santa Clara,CA,United States,29600.0,NVIDIA Corporation provides graphics and compu...,0.059350
2,NMS,MSFT,Microsoft Corporation,Microsoft Corporation,Technology,Software - Infrastructure,436.60,3246068596736,1.365520e+11,0.160,Redmond,WA,United States,228000.0,Microsoft Corporation develops and supports so...,0.058401
3,NMS,AMZN,"Amazon.com, Inc.","Amazon.com, Inc.",Consumer Cyclical,Internet Retail,224.92,2365033807872,1.115830e+11,0.110,Seattle,WA,United States,1551000.0,"Amazon.com, Inc. engages in the retail sale of...",0.042550
4,NMS,GOOGL,Alphabet Inc.,Alphabet Inc.,Communication Services,Internet Content & Information,191.41,2351625142272,1.234700e+11,0.151,Mountain View,CA,United States,181269.0,Alphabet Inc. offers various products and plat...,0.042309


In [21]:
df_raw.describe()

,Currentprice,Marketcap,Ebitda,Revenuegrowth,Fulltimeemployees,Weight
count,502.000000,5.020000e+02,4.730000e+02,499.000000,4.930000e+02,502.000000
mean,217.893685,1.107222e+11,7.045286e+09,0.070541,5.782761e+04,0.001992
std,489.205027,3.440074e+11,1.624213e+10,0.180247,1.395988e+05,0.006189
min,9.400000,4.664099e+09,-3.991000e+09,-0.602000,2.800000e+01,0.000084
25%,68.547500,1.933417e+10,1.625000e+09,0.002000,1.020000e+04,0.000348
50%,118.365000,3.706396e+10,2.952748e+09,0.051000,2.169000e+04,0.000667
75%,227.667500,7.830565e+10,6.020000e+09,0.109000,5.510000e+04,0.001409
max,8276.780000,3.846820e+12,1.495470e+11,1.632000,2.100000e+06,0.069209


In [22]:
df_raw.shape

(502, 16)

In [23]:
df_raw.dtypes

Exchange                   str
Symbol                     str
Shortname                  str
Longname                   str
Sector                     str
Industry                   str
Currentprice           float64
Marketcap                int64
Ebitda                 float64
Revenuegrowth          float64
City                       str
State                      str
Country                    str
Fulltimeemployees      float64
Longbusinesssummary        str
Weight                 float64
dtype: object

#### Observation
Text columns are stored as str, numbers as float64 or int64. pandas 3 uses its own data type for text, and not object.

In [24]:
# Why is Fulltimeemployees a float and not an integer?
employees = df_raw["Fulltimeemployees"]
print("Data type:", employees.dtype)


Data type: float64


In [25]:
employees.head()

0     164000.0
1      29600.0
2     228000.0
3    1551000.0
4     181269.0
Name: Fulltimeemployees, dtype: float64

In [26]:
print("Missing values:", employees.isna().sum())

# This explains why Fulltimeemployees is a float and not an integer: there are missing values.

Missing values: 9


In [27]:
# Companies with missing Fulltimeemployees values
df_raw.loc[employees.isna(), ["Symbol", "Shortname", "Sector"]]


,Symbol,Shortname,Sector
13,V,Visa Inc.,Financial Services
97,SBUX,Starbucks Corporation,Consumer Cyclical
206,DHI,"D.R. Horton, Inc.",Consumer Cyclical
257,RMD,ResMed Inc.,Healthcare
274,RJF,"Raymond James Financial, Inc.",Financial Services
384,SMCI,"Super Micro Computer, Inc.",Technology
430,FFIV,"F5, Inc.",Technology
464,SOLV,Solventum Corporation,Healthcare
501,AMTM,"Amentum Holdings, Inc.",Industrials


### Data Types Explanation
502 rows and 16 columns. Text columns are strings, numbers are float64 or int64. The column Fulltimeemployees is a float only because it has missing values. The rest does not show any anomalies compared to other datasets.


**What the columns mean** (from the Kaggle page)

| Column | Meaning |
|---|---|
| Exchange | Stock exchange (NYQ = New York Stock Exchange, NMS  = Nasdaq) where the stocks are negociated |
| Symbol | Stock Symbol (ticker) |
| Shortname, Longname | Company name |
| Sector, Industry | Sector and industry where the company operates |
| Currentprice | Share price in USD |
| Marketcap | market value of the company in USD |
| Ebitda | Earnings before interest, taxes, depreciation and amortization |
| Revenuegrowth | Revenue growth, compared to the same quarter 1 year before (cross-checked with quarterly reports of large companies) |
| City, State, Country | City, State and Country of the headquater |
| Fulltimeemployees | Number of full-time employees |
| Longbusinesssummary | Short description of the business |
| Weight | Percentage of participation on the S&P index (marketcap %) |

The file has no date column. I took 2024-12-20 as the date of the snapshot because it is the last trading day in the price files of the same version (Version 1023, published 2024-12-23).

---

## 3. Data Investigation
Before I elaborate on my research questions, I check missing values, duplicates, strange values categories.

### 3.1 Missing Values

In [30]:
# Building a reusable function for profile_missing, this function will be reused after cleaning and imputing the dataset.
def profile_missing(frame: pd.DataFrame) -> pd.DataFrame:
    """Returns the number and percent of missing values for every column that has gaps."""
    counts = frame.isna().sum()
    counts = counts[counts > 0].sort_values(ascending=False)
    return pd.DataFrame({"missing": counts, "percent": (counts / len(frame) * 100).round(1)})

In [31]:
profile_missing(df_raw)

,missing,percent
Ebitda,29,5.8
State,20,4.0
Fulltimeemployees,9,1.8
Revenuegrowth,3,0.6


In [32]:
# Rows without Ebitda, counted by sector
missing_ebitda = df_raw[df_raw["Ebitda"].isna()]
missing_ebitda["Sector"].value_counts()

Sector
Financial Services    29
Name: count, dtype: int64

In [33]:
# Companies without revenue growth
missing_growth = df_raw[df_raw["Revenuegrowth"].isna()]
missing_growth[["Symbol", "Shortname", "Sector"]]


,Symbol,Shortname,Sector
55,VZ,Verizon Communications Inc.,Communication Services
109,AMT,American Tower Corporation (REI,Real Estate
359,WDC,Western Digital Corporation,Technology


In [34]:
# Rows without State, counted by country
missing_state = df_raw[df_raw["State"].isna()]
missing_state["Country"].value_counts()


Country
Ireland           10
United Kingdom     3
Switzerland        3
Bermuda            2
Netherlands        1
Singapore          1
Name: count, dtype: int64

In [35]:
# Companies without an employee number
missing_employees = df_raw[df_raw["Fulltimeemployees"].isna()]
missing_employees[["Symbol", "Shortname", "Sector"]]


,Symbol,Shortname,Sector
13,V,Visa Inc.,Financial Services
97,SBUX,Starbucks Corporation,Consumer Cyclical
206,DHI,"D.R. Horton, Inc.",Consumer Cyclical
257,RMD,ResMed Inc.,Healthcare
274,RJF,"Raymond James Financial, Inc.",Financial Services
384,SMCI,"Super Micro Computer, Inc.",Technology
430,FFIV,"F5, Inc.",Technology
464,SOLV,Solventum Corporation,Healthcare
501,AMTM,"Amentum Holdings, Inc.",Industrials


In [36]:
missing_employees["Sector"].value_counts()

Sector
Financial Services    2
Consumer Cyclical     2
Healthcare            2
Technology            2
Industrials           1
Name: count, dtype: int64

#### Observation

- Ebitda (29 missing): all of them are in Financial Services. I researched and banks and insurers do not report this number, so the gap is not random. I do not use this column.
- State (20 missing): all of these companies have their headquarters outside the US, most of them in Ireland. They have no US state, so this is expected.
- Fulltimeemployees: (9 missing): the gaps are spread over five sectors, so there is no pattern. The column is only a float because of these gaps. I do not use it in the analysis.
- Revenuegrowth: (3 missing): Verizon, American Tower and Western Digital. This is my outcome, so I drop these 3 companies from the test. 3 of 502 rows (0.6 %) is too few to change the result, but it should be documented (Lusa et al., 2024).


### 3.2 Dublicates


In [39]:
print("Duplicate rows:", df_raw.duplicated().sum())
print("Duplicate symbols:", df_raw["Symbol"].duplicated().sum())

Duplicate rows: 0
Duplicate symbols: 0


In [40]:
# Also checking for shared classes
share_classes = df_raw[df_raw["Longname"].duplicated(keep=False)]
share_classes[["Symbol", "Longname", "Sector", "Marketcap", "Revenuegrowth"]]

,Symbol,Longname,Sector,Marketcap,Revenuegrowth
4,GOOGL,Alphabet Inc.,Communication Services,2351625142272,0.151
5,GOOG,Alphabet Inc.,Communication Services,2351623045120,0.151
345,FOXA,Fox Corporation,Communication Services,21946050560,0.111
346,FOX,Fox Corporation,Communication Services,21941315584,0.111
410,NWS,News Corporation,Communication Services,16374949888,0.031
411,NWSA,News Corporation,Communication Services,16357357568,0.031


#### Observations

No duplicate rows but the companies Alphabet, Fox and News Corp are in the file multiple times, because they have two types of shares (share classes). Both rows have the same sector and the same growth. Counting them twice would be wrong, we will handle this in the cleaning phase.

### 3.3 Checking if weight is marketcap in  %


In [48]:
# Calculating implied weight based on market capitalization
implied_weight = df_raw["Marketcap"] / df_raw["Marketcap"].sum()
# Cheking if the actual weight matches the implied weight for the largest companies
largest_companies = df_raw.nlargest(10, "Marketcap")
largest_companies[["Symbol", "Marketcap", "Weight"]].assign(ImpliedWeight=implied_weight[largest_companies.index])


,Symbol,Marketcap,Weight,ImpliedWeight
0,AAPL,3846819807232,0.069209,0.069209
1,NVDA,3298803056640,0.059350,0.059350
2,MSFT,3246068596736,0.058401,0.058401
3,AMZN,2365033807872,0.042550,0.042550
4,GOOGL,2351625142272,0.042309,0.042309
5,GOOG,2351623045120,0.042309,0.042309
6,META,1477457739776,0.026581,0.026581
7,TSLA,1351627833344,0.024317,0.024317
8,AVGO,1031217348608,0.018553,0.018553
9,BRK-B,978776031232,0.017609,0.017609


#### Observation
Yes Weight is just Marketcap divided by the total, so it does not add nothing new.

### 3.4 Checking target variable for plausability

I will use different checks to make sure we dont have any strange and unexplainable values in the outcome variable.

In [49]:
# Step 1: What is the lowest and the highest revenue growth?
growth = df_raw["Revenuegrowth"]
print("Lowest value:", growth.min())
print("Highest value:", growth.max())

Lowest value: -0.602
Highest value: 1.632


In [51]:
# Step 2: How many companies had less revenue than one year before?
print("Negative values:", (growth < 0).sum())

Negative values: 124


In [54]:
# Step 3: Is 0 used as a placeholder for missing data?
# Checking against missing values again and getting the exact companies
print("Missing values:", growth.isna().sum())
print("Exact zeros:", (growth == 0).sum())

Missing values: 3
Exact zeros: 0


In [55]:
# Step 4: Checking for data collection errors, by printing out the 5 largest revenue growth values for plausability
df_raw.nlargest(5, "Revenuegrowth")[["Symbol", "Shortname", "Sector", "Revenuegrowth"]]

,Symbol,Shortname,Sector,Revenuegrowth
307,SW,Smurfit WestRock plc,Consumer Cyclical,1.632
384,SMCI,"Super Micro Computer, Inc.",Technology,1.430
223,PRU,"Prudential Financial, Inc.",Financial Services,1.334
1,NVDA,NVIDIA Corporation,Technology,1.224
96,MU,"Micron Technology, Inc.",Technology,0.933


In [56]:
# Step 5: Doing the same for the lowest revenue growth values
df_raw.nsmallest(5, "Revenuegrowth")[["Symbol", "Shortname", "Sector", "Revenuegrowth"]]

,Symbol,Shortname,Sector,Revenuegrowth
403,KEY,KeyCorp,Financial Services,-0.602
287,MCHP,Microchip Technology Incorporat,Technology,-0.484
471,ALB,Albemarle Corporation,Basic Materials,-0.414
167,AFL,AFLAC Incorporated,Financial Services,-0.404
392,PFG,Principal Financial Group Inc,Financial Services,-0.345


#### Observations

- The values go from −0.602 to 1.632, no value is exactly 0, and 124 values are negative.
- I checked a few values against the quarterly reports of the companies. They match the revenue growth of one quarter compared with the same quarter one year before. So 124 companies had less revenue than one year before.

Because a few values are very high or very low, I will also check the result with the median.



### 3.5 Checking Categories

In [ ]:
# Step 1: How many companies are in each sector? Is any sector too small for a group?
df_raw["Sector"].value_counts()

Sector
Technology                82
Industrials               70
Financial Services        67
Healthcare                62
Consumer Cyclical         55
Consumer Defensive        37
Utilities                 32
Real Estate               31
Communication Services    22
Energy                    22
Basic Materials           22
Name: count, dtype: int64

In [ ]:
# Step 2: On which stock exchanges are the companies?
df_raw["Exchange"].value_counts()

Exchange
NYQ    348
NMS    152
BTS      1
NGM      1
Name: count, dtype: int64

In [60]:
# Step 3: In which countries are the companies located?
df_raw["Country"].value_counts()

Country
United States     481
Ireland            10
United Kingdom      3
Switzerland         3
Bermuda             2
Netherlands         1
Canada              1
Singapore           1
Name: count, dtype: int64

In [62]:
# Step 4: In which sectors are the largest companies?
df_raw.nlargest(10, "Marketcap")[["Symbol", "Shortname", "Sector"]]


,Symbol,Shortname,Sector
0,AAPL,Apple Inc.,Technology
1,NVDA,NVIDIA Corporation,Technology
2,MSFT,Microsoft Corporation,Technology
3,AMZN,"Amazon.com, Inc.",Consumer Cyclical
4,GOOGL,Alphabet Inc.,Communication Services
5,GOOG,Alphabet Inc.,Communication Services
6,META,"Meta Platforms, Inc.",Communication Services
7,TSLA,"Tesla, Inc.",Consumer Cyclical
8,AVGO,Broadcom Inc.,Technology
9,BRK-B,Berkshire Hathaway Inc. New,Financial Services


#### Observation

No sector is too small, because the smallest has 22 companies! Technology is the biggest with 82. Exchange and country are not used in the analysis.

Good to know: the sector labels in this file put Alphabet and Meta in Communication Services and Amazon in Consumer Cyclical. So "Technology" in this file is smaller than what people usually mean with "big tech"!

---

## 4. Data Cleaning
Based on the data check in Section 3 I clean the data step by step. The raw data in df_raw stay unchanged for future reference.

In [63]:
# Step 1: Keep one row per company.
# Alphabet, Fox and News Corp have two share classes (see 3.2). Both rows of a company have the same sector, the same growth and almost the same market value (the value of the whole company), so I keep the first row.

companies = df_raw.drop_duplicates(subset="Longname", keep="first").copy()
print("Rows before:", len(df_raw))
print("Companies after:", len(companies))

Rows before: 502
Companies after: 499


In [64]:
# Step 2: Which rows did I remove?
df_raw[~df_raw["Symbol"].isin(companies["Symbol"])][["Symbol", "Longname"]]

,Symbol,Longname
5,GOOG,Alphabet Inc.
346,FOX,Fox Corporation
411,NWSA,News Corporation


In [66]:
# Step 3: Check that every company is now in the data only once
companies["Longname"].is_unique

True

#### Observation

In total we end up with only 499 companies in the dataset, which is strange since the S&P500 consists of 500, so potentially there was a merger, or some other event happening.

In [67]:
# Step 4: Drop the 3 companies without revenue growth. My test will need a value for every company.
df = companies.dropna(subset=["Revenuegrowth"]).copy()
print("Companies with revenue growth:", len(df))

Companies with revenue growth: 496


In [68]:
# Step 5: Revenue growth in percent (0.05 -> 5.0) will be much easier to read in tables and figures
df["revenue_growth_pct"] = df["Revenuegrowth"] * 100
df[["Symbol", "Revenuegrowth", "revenue_growth_pct"]].head()

,Symbol,Revenuegrowth,revenue_growth_pct
0,AAPL,0.061,6.1
1,NVDA,1.224,122.4
2,MSFT,0.160,16.0
3,AMZN,0.110,11.0
4,GOOGL,0.151,15.1


In [69]:
# Step 6: Two groups for the test Technology and all other sectors
df["group"] = np.where(df["Sector"] == "Technology", "Technology", "Other sectors")
df["group"].value_counts()

group
Other sectors    415
Technology        81
Name: count, dtype: int64

### Result of the cleaning
- 502 rows --> 499 companies (one row per company): dataset: companies --> 496 companies with revenue growth: dataset: df (main dataset)
- 81 in Technology and 415 in other sectors.


---

## 5. Descriptive Statistics

### 5.1 Summary statistics
Only the two numeric columns I use: market value (in billion USD) and revenue growth (in %).

In [77]:
# Step 1: Market value in billion USD (1 billion = 1000000000), so the numbers are easier to read, and additionally round the summary statistics for better readability
companies["marketcap_bn"] = companies["Marketcap"] / 1000000000
round(companies["marketcap_bn"].describe(), 2)

count     499.00
mean      106.60
std       330.03
min         4.66
25%        19.35
50%        37.19
75%        78.07
max      3846.82
Name: marketcap_bn, dtype: float64

In [ ]:
# Step 2: Skewness of the market value (0 = symmetric, positive / negative skewed)
print(round(companies["marketcap_bn"].skew(), 2))

8.16


In [80]:
# Step 3: Revenue growth in % (the column from Data Cleaning, Step 5)
round(df["revenue_growth_pct"].describe(), 2)

count    496.00
mean       7.04
std       18.07
min      -60.20
25%        0.12
50%        5.00
75%       10.90
max      163.20
Name: revenue_growth_pct, dtype: float64

In [82]:
# Step 4: Skewness of revenue growth
print(round(df["revenue_growth_pct"].skew(), 2))

3.82


#### Observation

Both columns are right-skewed (skewness 8.16 and 3.82), a few very big values seem to pull the mean higher than the median. The median market value is USD 37.19 billions (50%), the mean USD 106.60 billions. 
The typical company grew its revenue by 5.0 %, the mean is 7.0 %, and 124 of the 496 companies (25 %) had falling revenue (0.12). 
That is why I report the median and the quartiles next to the mean, like Lusa et al. (2024).

### 5.2 Companies and market values per sector
An ETF invests by market value, so a sectors share of the market value is the share of my money in it.

In [ ]:
# Step 1: How many companies are in each sector? Recap from above
sector_count = companies["Sector"].value_counts()
sector_count

Sector
Technology                82
Industrials               70
Financial Services        67
Healthcare                62
Consumer Cyclical         55
Consumer Defensive        37
Utilities                 32
Real Estate               31
Energy                    22
Basic Materials           22
Communication Services    19
Name: count, dtype: int64

In [87]:
# Step 2: Share of the companies in % (companies in the sector / all 499 companies)
share_of_companies = sector_count / 499 * 100
round(share_of_companies, 2)

Sector
Technology                16.43
Industrials               14.03
Financial Services        13.43
Healthcare                12.42
Consumer Cyclical         11.02
Consumer Defensive         7.41
Utilities                  6.41
Real Estate                6.21
Energy                     4.41
Basic Materials            4.41
Communication Services     3.81
Name: count, dtype: float64

In [ ]:
# Step 3: Total market value of each sector in billions
sector_value = companies.groupby("Sector")["marketcap_bn"].sum()
round(sector_value, 2)

Sector
Basic Materials             864.55
Communication Services     5414.12
Consumer Cyclical          6372.35
Consumer Defensive         3253.84
Energy                     1598.67
Financial Services         7108.16
Healthcare                 5190.43
Industrials                3764.60
Real Estate                1081.49
Technology                17287.01
Utilities                  1257.37
Name: marketcap_bn, dtype: float64

In [91]:
# Step 4: Share of the total market value in % (market value of the sector / market value of all companies)
share_of_value = sector_value / companies["marketcap_bn"].sum() * 100
round(share_of_value, 2)

Sector
Basic Materials            1.63
Communication Services    10.18
Consumer Cyclical         11.98
Consumer Defensive         6.12
Energy                     3.01
Financial Services        13.36
Healthcare                 9.76
Industrials                7.08
Real Estate                2.03
Technology                32.50
Utilities                  2.36
Name: marketcap_bn, dtype: float64

In [95]:
# Step 5 Both shares in only one table, sorted by the share of market value
# Create a table with both shares side by side
concentration = pd.DataFrame()

# Add the share of companies and share of market value to the table
concentration["share_of_companies_pct"] = share_of_companies
concentration["share_of_market_value_pct"] = share_of_value

# Sort the table by the share of market value in descending order
concentration = concentration.sort_values("share_of_market_value_pct", ascending=False)
concentration

,share_of_companies_pct,share_of_market_value_pct
Sector,,
Technology,16.432866,32.498901
Financial Services,13.426854,13.363065
Consumer Cyclical,11.022044,11.979771
Communication Services,3.807615,10.178340
Healthcare,12.424850,9.757805
Industrials,14.028056,7.077302
Consumer Defensive,7.414830,6.117088
Energy,4.408818,3.005440
Utilities,6.412826,2.363805


In [96]:
# Step 6: The 10 biggest companies by market value (in billion USD)
top10 = companies.nlargest(10, "marketcap_bn")
top10[["Symbol", "Shortname", "Sector", "marketcap_bn"]]

,Symbol,Shortname,Sector,marketcap_bn
0,AAPL,Apple Inc.,Technology,3846.819807
1,NVDA,NVIDIA Corporation,Technology,3298.803057
2,MSFT,Microsoft Corporation,Technology,3246.068597
3,AMZN,"Amazon.com, Inc.",Consumer Cyclical,2365.033808
4,GOOGL,Alphabet Inc.,Communication Services,2351.625142
6,META,"Meta Platforms, Inc.",Communication Services,1477.457740
7,TSLA,"Tesla, Inc.",Consumer Cyclical,1351.627833
8,AVGO,Broadcom Inc.,Technology,1031.217349
9,BRK-B,Berkshire Hathaway Inc. New,Financial Services,978.776031
10,WMT,Walmart Inc.,Consumer Defensive,740.999889


In [ ]:
# Step 7: Share of the 10 biggest companies in the total market value
top10_share = top10["marketcap_bn"].sum() / companies["marketcap_bn"].sum() * 100
print(round(top10_share, 2))

38.89


In [101]:
# Step 8: Share of these 10 companies in the number of companies
print(round(10 / 499 * 100, 2))

2.0


#### Observation

- The index is a lot less spread out than 500 companies sounds. 
- Technology is 16.4 % of the companies but they have 32.5 % of the market value, twice as much. 
- The 10 biggest companies are 2.0 % of the companies but hold 38.9 %. 
- Eight of them are tech companies (Apple, NVIDIA, Microsoft, Amazon, Alphabet, Meta, Tesla, Broadcom) even if the dataset puts four of them in other sectors. 
- This big weight would make more sense if technology companies also grow faster. That is what I will test later on!

In [102]:
# DESCRIBE by group
group_summary = df.groupby("group")["revenue_growth_pct"].describe()
group_summary

,count,mean,std,min,25%,50%,75%,max
group,,,,,,,,
Other sectors,415.0,6.150602,15.881143,-60.2,-0.2,4.5,9.9,163.2
Technology,81.0,11.582716,26.287620,-48.4,2.5,7.1,16.6,143.0


#### Observation

Technology companies grow faster on average (mean 11.6 % vs. 6.2 %, 5.4 percent points more), but the medians differ by only 2.6 points (7.1 % vs. 4.5 %): a few very fast growing technology companies move the tech mean up. 
Tech also varies much more (standard deviation 26.3 vs. 15.9 points). This is something most investors are not aware off, you buy a big chunk of very fluctating tech!

---

## 6. Visualization

---

## 7. Hypothesis Testing

---

## 8. Summary